In [1]:
!pip install transformers ffmpeg-python librosa noisereduce ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 994.0/994.0 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 80.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [8]:
!pip install protobuf==4.25.6 mediapipe grpcio-status --force-reinstall

  Using cached protobuf-4.25.6-cp37-abi3-manylinux2014_x86_64.whl.metadata (541 bytes)
  Using cached mediapipe-0.10.21-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (9.7 kB)
  Using cached grpcio_status-1.71.0-py3-none-any.whl.metadata (1.1 kB)
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached sounddevice-0.5.1-py3-none-any.whl.metadata (1.4 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [17]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Use GPU 0 explicitly

import tensorflow as tf
print("Available GPUs:", tf.config.list_physical_devices('GPU'))

Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


C:\Users\izall\anaconda3\envs\gpu-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: NVIDIA GeForce RTX 4060 Laptop GPU


In [ ]:
import cv2
import mediapipe as mp
import os
import numpy as np
import pandas as pd
import glob
from transformers import ViTFeatureExtractor, ViTModel, Wav2Vec2FeatureExtractor, Wav2Vec2Model
import torch
import ffmpeg
import librosa
import noisereduce as nr
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

# Set environment variable for GPU usage
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {torch.cuda.get_device_name(device)}")

# Initialize MediaPipe Face Mesh
mp_face_mesh = mp.solutions.face_mesh

# Initialize ViT and Wav2Vec2 models on GPU
feature_extractor = ViTFeatureExtractor.from_pretrained('google/vit-base-patch16-224-in21k')
vit_model = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k').to(device)  # Move to GPU

wav2vec_model_name = "facebook/wav2vec2-xls-r-300m"
wav2vec_feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(wav2vec_model_name)
wav2vec_model = Wav2Vec2Model.from_pretrained(wav2vec_model_name).to(device)  # Move to GPU

# Download and load YOLOv8-Face-Detection model
model_path = hf_hub_download(repo_id="arnabdhar/YOLOv8-Face-Detection", filename="model.pt")
yolo_face_model = YOLO(model_path)

def process_and_extract_features(frame):
    """Process frame and extract features with MediaPipe first, then YOLO-Face fallback."""
    try:
        # Initialize MediaPipe Face Mesh
        with mp_face_mesh.FaceMesh(
            static_image_mode=True,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.3,
            min_tracking_confidence=0.3
        ) as face_mesh:

            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h, w, _ = frame.shape

            # Attempt face detection with MediaPipe
            results = face_mesh.process(image_rgb)
            if results.multi_face_landmarks:
                # If face detected, process with ViT on GPU
                inputs = feature_extractor(images=image_rgb, return_tensors="pt").to(device)
                outputs = vit_model(**inputs)
                features = outputs.last_hidden_state.mean(dim=1).detach().cpu().numpy()
                return features.flatten()

            # Fallback to YOLOv8-Face if MediaPipe fails
            print("MediaPipe failed. Trying YOLOv8-Face for detection...")
            results = yolo_face_model.predict(frame, conf=0.5, verbose=False)
            if results and hasattr(results[0], 'boxes') and results[0].boxes is not None:
                boxes = results[0].boxes.xyxy.cpu().numpy()
                if len(boxes) > 0:
                    x1, y1, x2, y2 = boxes[0].astype(int)
                    cropped_face = frame[y1:y2, x1:x2]
                    if cropped_face.size > 0:
                        cropped_rgb = cv2.cvtColor(cropped_face, cv2.COLOR_BGR2RGB)
                        inputs = feature_extractor(images=cropped_rgb, return_tensors="pt").to(device)
                        outputs = vit_model(**inputs)
                        features = outputs.last_hidden_state.mean(dim=1).detach().cpu().numpy()
                        return features.flatten()
            print("Face detection failed for this frame.")
            return None

    except Exception as e:
        print(f"Error processing frame: {str(e)}")
        return None

def extract_audio(video_path, output_audio_path="temp_audio.wav"):
    """Extract audio from video using FFmpeg."""
    (
        ffmpeg.input(video_path)
        .output(output_audio_path, ac=1, ar=16000)
        .overwrite_output()
        .run(quiet=True)
    )
    return output_audio_path

def get_xlsr_embeddings(audio_path):
    """Extract XLS-R embeddings from audio."""
    audio, sr = librosa.load(audio_path, sr=16000)
    audio = librosa.util.normalize(audio)
    audio = nr.reduce_noise(y=audio, sr=sr)

    inputs = wav2vec_feature_extractor(
        audio,
        return_tensors="pt",
        sampling_rate=16000,
        padding="max_length",
        max_length=16000 * 10,
        truncation=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}  
    with torch.no_grad():
        outputs = wav2vec_model(**inputs)

    return outputs.last_hidden_state.mean(dim=1).cpu().squeeze().numpy()

def process_all_media_in_folder(folder_path):
    """Process all media files in the specified folder."""
    video_files = glob.glob(os.path.join(folder_path, '**', '*.mp4'), recursive=True)
    audio_data = []
    frame_data = []

    for video_path in video_files:
        print(f"Processing video: {video_path}")
        video_name = os.path.basename(video_path)

        # Process audio
        audio_path = extract_audio(video_path)
        audio_features = get_xlsr_embeddings(audio_path)
        audio_data.append({'Video File': video_name, 'Features': audio_features.tolist()})
        os.remove(audio_path)

        # Process video frames
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        
        # Ensure fps is not zero to avoid ZeroDivisionError
        if fps == 0:
            print(f"Skipping file due to zero FPS: {video_path}")
            cap.release()
            continue

        frame_interval = int(fps / 24)  # Process every 24th frame
        frame_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if frame_count % frame_interval == 0:
                features = process_and_extract_features(frame)
                if features is not None:
                    frame_data.append({
                        'Video File': video_name,
                        'Frame Index': frame_count,
                        'Features': features.tolist()
                    })
            frame_count += 1
        cap.release()

    # Save audio features
    audio_df = pd.DataFrame(audio_data)
    audio_df.to_csv('audio_features.csv', index=False)
    print("Audio features saved to audio_features.csv")

    # Save frame features
    frame_df = pd.DataFrame(frame_data)
    frame_df.to_csv('frame_features.csv', index=False)
    print("Frame features saved to frame_features.csv")

# Run the pipeline on your folder
folder_path = "PolyGlotFake/train"
process_all_media_in_folder(folder_path)

Using device: NVIDIA GeForce RTX 4060 Laptop GPU
Processing video: PolyGlotFake/train\fake\ar_10_to_en_MicroTts.mp4
Processing video: PolyGlotFake/train\fake\ar_10_to_en_Xtts.mp4
Processing video: PolyGlotFake/train\fake\ar_10_to_es_MicroTts.mp4
Processing video: PolyGlotFake/train\fake\ar_10_to_es_Xtts.mp4
Processing video: PolyGlotFake/train\fake\ar_10_to_fr_MicroTts.mp4
Processing video: PolyGlotFake/train\fake\ar_10_to_ja_MicroTts.mp4
Processing video: PolyGlotFake/train\fake\ar_10_to_ru_Xtts.mp4
Processing video: PolyGlotFake/train\fake\ar_10_to_zh_MicroTts.mp4
Processing video: PolyGlotFake/train\fake\ar_11_to_en_MicroTts.mp4
Processing video: PolyGlotFake/train\fake\ar_11_to_es_MicroTts.mp4
Processing video: PolyGlotFake/train\fake\ar_11_to_es_Xtts.mp4
Processing video: PolyGlotFake/train\fake\ar_11_to_fr_MicroTts.mp4
Processing video: PolyGlotFake/train\fake\ar_11_to_fr_Xtts.mp4
Processing video: PolyGlotFake/train\fake\ar_11_to_ru_MicroTts.mp4
Processing video: PolyGlotFake/tr

In [ ]:
import os
import cv2
import glob
import ffmpeg
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
import torch
import noisereduce as nr
import mediapipe as mp
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Model

# Allow GPU memory growth for TensorFlow
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    try:
        for device in physical_devices:
            tf.config.experimental.set_memory_growth(device, True)
        print("GPU memory growth allowed for TensorFlow.")
    except RuntimeError as e:
        print(f"Error setting memory growth: {e}")

# Set PyTorch device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using PyTorch device: {torch.cuda.get_device_name(device)}")

# Load the VGG16 model pre-trained on ImageNet and move to GPU
base_model = VGG16(weights='imagenet')
model = Model(inputs=base_model.input, outputs=base_model.get_layer('fc1').output)

# Download and load YOLOv8-Face-Detection model
model_path = hf_hub_download(repo_id="arnabdhar/YOLOv8-Face-Detection", filename="model.pt")
yolo_face_model = YOLO(model_path)

def process_and_extract_features(frame):
    """Process frame and extract features using MediaPipe first, then YOLOv8-Face fallback"""
    try:
        # Initialize MediaPipe Face Mesh for this frame
        with mp.solutions.face_mesh.FaceMesh(
            static_image_mode=True,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.3,
            min_tracking_confidence=0.3
        ) as face_mesh:

            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h, w, _ = frame.shape

            # First try with MediaPipe
            results = face_mesh.process(image_rgb)

            if results.multi_face_landmarks:
                for face_landmarks in results.multi_face_landmarks:
                    # Extract bounding box from landmarks
                    x_min = int(min([lm.x for lm in face_landmarks.landmark]) * w)
                    y_min = int(min([lm.y for lm in face_landmarks.landmark]) * h)
                    x_max = int(max([lm.x for lm in face_landmarks.landmark]) * w)
                    y_max = int(max([lm.y for lm in face_landmarks.landmark]) * h)

                    # Extract the nose tip and lower face region
                    nose_tip = face_landmarks.landmark[1]
                    nose_tip_y = int(nose_tip.y * h)
                    lower_face_crop = frame[nose_tip_y:y_max, x_min:x_max]

                    if lower_face_crop.size > 0:
                        lower_face_crop = cv2.resize(lower_face_crop, (224, 224))
                        lower_face_crop = np.expand_dims(lower_face_crop, axis=0)
                        lower_face_crop = preprocess_input(lower_face_crop)
                        features = model.predict(lower_face_crop)  # TensorFlow uses GPU automatically
                        return features.flatten()

            # If MediaPipe failed, try YOLOv8-Face
            print("MediaPipe failed. Trying YOLOv8-Face for detection...")
            results = yolo_face_model.predict(frame, conf=0.5, verbose=False)

            if results and hasattr(results[0], 'boxes') and results[0].boxes is not None:
                boxes = results[0].boxes.xyxy.cpu().numpy()

                if len(boxes) > 0:
                    # Extract the first detected face box
                    x1, y1, x2, y2 = boxes[0].astype(int)
                    cropped_face = frame[y1:y2, x1:x2]

                    if cropped_face.size > 0:
                        # Try MediaPipe again on the cropped face
                        cropped_rgb = cv2.cvtColor(cropped_face, cv2.COLOR_BGR2RGB)
                        results_cropped = face_mesh.process(cropped_rgb)

                        if results_cropped.multi_face_landmarks:
                            ch, cw, _ = cropped_face.shape
                            for face_landmarks in results_cropped.multi_face_landmarks:
                                x_min = int(min([lm.x for lm in face_landmarks.landmark]) * cw)
                                x_max = int(max([lm.x for lm in face_landmarks.landmark]) * cw)
                                y_max = int(max([lm.y for lm in face_landmarks.landmark]) * ch)
                                nose_tip = face_landmarks.landmark[1]
                                nose_tip_y = int(nose_tip.y * ch)

                                lower_face_crop = cropped_face[nose_tip_y:y_max, x_min:x_max]
                                if lower_face_crop.size > 0:
                                    lower_face_crop = cv2.resize(lower_face_crop, (224, 224))
                                    lower_face_crop = np.expand_dims(lower_face_crop, axis=0)
                                    lower_face_crop = preprocess_input(lower_face_crop)
                                    features = model.predict(lower_face_crop)  
                                    return features.flatten()
                            print("Retry with cropped face also failed.")
            else:
                print("YOLOv8-Face could not detect a face.")

    except Exception as e:
        print(f"Error processing frame: {str(e)}")

    return None

def extract_audio(video_path, output_audio_path="temp_audio.wav"):
    ffmpeg.input(video_path).output(output_audio_path, ac=1, ar=16000).overwrite_output().run(quiet=True)
    return output_audio_path

def get_xlsr_embeddings(audio_path):
    model_name = "facebook/wav2vec2-xls-r-300m"
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
    model = Wav2Vec2Model.from_pretrained(model_name).to(device)  # Move model to GPU

    audio, sr = librosa.load(audio_path, sr=16000)
    audio = librosa.util.normalize(audio)
    audio = nr.reduce_noise(y=audio, sr=sr)

    inputs = feature_extractor(
        audio,
        return_tensors="pt",
        sampling_rate=16000,
        padding="max_length",
        max_length=16000 * 10,
        truncation=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}  # Move inputs to GPU

    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.last_hidden_state.mean(dim=1).cpu().squeeze().numpy()

def process_all_media_in_folder(folder_path):
    """Process all media files in the specified folder."""
    video_files = glob.glob(os.path.join(folder_path, '**', '*.mp4'), recursive=True)

    audio_data = []
    frame_data = []

    for video_path in video_files:
        print(f"Processing video: {video_path}")
        video_name = os.path.basename(video_path)

        # Extract and process audio
        audio_path = extract_audio(video_path)
        audio_features = get_xlsr_embeddings(audio_path)
        audio_data.append({'Video File': video_name, 'Features': audio_features.tolist()})
        os.remove(audio_path)

        # Process video frames
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)

        # Ensure fps is not zero to avoid ZeroDivisionError
        if fps == 0:
            print(f"Skipping file due to zero FPS: {video_path}")
            cap.release()
            continue

        frame_interval = int(fps / 24)  # Process every 24th frame
        frame_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if frame_count % frame_interval == 0:
                features = process_and_extract_features(frame)
                if features is not None:
                    frame_data.append({
                        'Video File': video_name,
                        'Frame Index': frame_count,
                        'Features': features.tolist()
                    })
            frame_count += 1
        cap.release()

    # Save audio features
    audio_df = pd.DataFrame(audio_data)
    audio_df.to_csv('audio_features.csv', index=False)
    print("Audio features saved to audio_features.csv")

    # Save frame features
    frame_df = pd.DataFrame(frame_data)
    frame_df.to_csv('frame_features.csv', index=False)
    print("Frame features saved to frame_features.csv")

# Run on your folder
folder_path = "/content/drive/MyDrive/data/testyolo"
process_all_media_in_folder(folder_path)

In [4]:
import cv2
import mediapipe as mp
import os
import numpy as np
import pandas as pd
import glob
from transformers import ViTFeatureExtractor, ViTModel, Wav2Vec2FeatureExtractor, Wav2Vec2Model
import torch
import torch.nn as nn
import ffmpeg
import librosa
import noisereduce as nr
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
from torch.utils.data import Dataset, DataLoader
from typing import Tuple, Dict, List, Any
from tqdm import tqdm
import csv
import concurrent.futures
from functools import partial

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {torch.cuda.get_device_name(device)}")

# Initialize models globally for reuse
mp_face_mesh = mp.solutions.face_mesh

# Mouth landmark indices (MediaPipe's 468-point face mesh)
MOUTH_LANDMARKS = [
    61, 185, 40, 39, 37, 0, 267, 269, 270, 409,
    291, 146, 91, 181, 84, 17, 314, 405, 321, 375,
    78, 191, 80, 81, 82, 13, 312, 311, 310, 415,
    95, 88, 178, 87, 14, 317, 402, 318, 324, 308
]

# Initialize models with caching
class AVFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.visual_extractor = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k').to(device).eval()
        self.audio_extractor = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-xls-r-300m").to(device).eval()
        self.vit_feature_extractor = ViTFeatureExtractor.from_pretrained('google/vit-base-patch16-224-in21k')
        self.wav2vec_feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-xls-r-300m")
        
        # Freeze models
        for param in self.visual_extractor.parameters():
            param.requires_grad = False
        for param in self.audio_extractor.parameters():
            param.requires_grad = False
            
    def forward(self, visual_input, audio_input):
        with torch.no_grad():
            visual_features = self.visual_extractor(**visual_input).last_hidden_state.mean(dim=1)
            audio_features = self.audio_extractor(**audio_input).last_hidden_state.mean(dim=1)
        return visual_features, audio_features

class AVFusionModel(nn.Module):
    def __init__(self, visual_dim=768, audio_dim=1024, hidden_dim=512):
        super().__init__()
        self.fusion = nn.Sequential(
            nn.Linear(visual_dim + audio_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
    def forward(self, visual_features, audio_features):
        return self.fusion(torch.cat([visual_features, audio_features], dim=1))

# Initialize models
feature_extractor = AVFeatureExtractor()
fusion_model = AVFusionModel().to(device).eval()
yolo_face_model = YOLO(hf_hub_download(repo_id="arnabdhar/YOLOv8-Face-Detection", filename="model.pt"))

def initialize_csv(output_file: str):
    """Initialize CSV file with headers"""
    with open(output_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            'video_file', 'frame_index', 'frame_time',
            'visual_features', 'audio_features', 'fused_features'
        ])

def append_to_csv(output_file: str, data: Dict[str, Any]):
    """Append a single row to the CSV file"""
    with open(output_file, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            data['video_file'],
            data['frame_index'],
            data['frame_time'],
            str(data['visual_features']),  # Convert lists to strings for CSV
            str(data['audio_features']),
            str(data['fused_features'])
        ])

def get_mouth_roi(face_landmarks: Any, frame: np.ndarray, padding: int = 20) -> np.ndarray:
    """Optimized mouth region extraction"""
    h, w = frame.shape[:2]
    coords = np.array([(int(lm.x * w), int(lm.y * h)) for lm in [face_landmarks.landmark[i] for i in MOUTH_LANDMARKS]])
    x_min, y_min = np.min(coords, axis=0) - padding
    x_max, y_max = np.max(coords, axis=0) + padding
    return frame[max(0,y_min):min(h,y_max), max(0,x_min):min(w,x_max)]

def extract_visual_features(frame: np.ndarray) -> Dict[str, torch.Tensor]:
    """Optimized visual feature extraction with batching support"""
    try:
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Initialize FaceMesh for each frame to avoid timestamp issues
        with mp_face_mesh.FaceMesh(
            static_image_mode=True,  # Treat each frame independently
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        ) as face_mesh:
            results = face_mesh.process(image_rgb)
            
            if results.multi_face_landmarks:
                mouth_roi = get_mouth_roi(results.multi_face_landmarks[0], frame)
                if mouth_roi.size > 0:
                    mouth_roi = cv2.resize(mouth_roi, (224, 224))
                    return feature_extractor.vit_feature_extractor(
                        images=cv2.cvtColor(mouth_roi, cv2.COLOR_BGR2RGB), 
                        return_tensors="pt"
                    ).to(device)
                    
        # Fallback to YOLO if MediaPipe fails
        results = yolo_face_model.predict(frame, conf=0.5, verbose=False, imgsz=320)
        if results and results[0].boxes is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            if len(boxes) > 0:
                x1, y1, x2, y2 = boxes[0].astype(int)
                cropped_face = frame[y1:y2, x1:x2]
                if cropped_face.size > 0:
                    with mp_face_mesh.FaceMesh(
                        static_image_mode=True,
                        max_num_faces=1,
                        refine_landmarks=True,
                        min_detection_confidence=0.5,
                        min_tracking_confidence=0.5
                    ) as face_mesh:
                        results = face_mesh.process(cv2.cvtColor(cropped_face, cv2.COLOR_BGR2RGB))
                        if results.multi_face_landmarks:
                            mouth_roi = get_mouth_roi(results.multi_face_landmarks[0], cropped_face)
                            if mouth_roi.size > 0:
                                mouth_roi = cv2.resize(mouth_roi, (224, 224))
                                return feature_extractor.vit_feature_extractor(
                                    images=cv2.cvtColor(mouth_roi, cv2.COLOR_BGR2RGB),
                                    return_tensors="pt"
                                ).to(device)
        return None
    except Exception as e:
        print(f"\nVisual processing error: {str(e)}")
        return None

def extract_audio_features(audio: np.ndarray, sr: int, frame_time: float, 
                         window_size: float = 0.2) -> Dict[str, torch.Tensor]:
    """Optimized audio feature extraction"""
    try:
        start = int(max(0, (frame_time - window_size/2) * sr))
        end = int(min(len(audio), (frame_time + window_size/2) * sr))
        segment = audio[start:end] if (end-start) > 0 else np.zeros(int(sr * window_size))
        
        return feature_extractor.wav2vec_feature_extractor(
            segment,
            return_tensors="pt",
            sampling_rate=sr,
            padding="max_length",
            max_length=int(sr * window_size),
            truncation=True
        ).to(device)
    except Exception as e:
        print(f"\nAudio processing error: {str(e)}")
        return None

def extract_audio(video_path: str) -> Tuple[np.ndarray, int]:
    """Faster audio extraction using FFmpeg pipe"""
    try:
        out, _ = (
            ffmpeg.input(video_path)
            .output('pipe:', format='wav', ac=1, ar=16000)
            .run(capture_stdout=True, quiet=True)
        )
        return np.frombuffer(out, np.int16).astype(np.float32) / 32768.0, 16000
    except Exception as e:
        print(f"\nError extracting audio: {str(e)}")
        return None, None

def process_video(video_path: str, max_duration: float = 5.0) -> List[Dict[str, Any]]:
    """Process a single video with optimized pipeline"""
    video_name = os.path.basename(video_path)
    samples = []
    
    # Extract audio once per video
    audio, sr = extract_audio(video_path)
    if audio is None:
        return []
    
    # Apply noise reduction to entire audio
    try:
        audio = nr.reduce_noise(y=audio, sr=sr)
    except Exception as e:
        print(f"\nNoise reduction failed: {str(e)}")
    
    # Process video frames
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0:
        cap.release()
        return []
    
    frame_interval = max(1, int(fps / 24))  # Target ~24fps
    max_frame = int(fps * max_duration)
    frame_count = 0
    
    while cap.isOpened() and frame_count < max_frame:
        ret, frame = cap.read()
        if not ret:
            break
            
        if frame_count % frame_interval == 0:
            frame_time = frame_count / fps
            visual_input = extract_visual_features(frame)
            audio_input = extract_audio_features(audio, sr, frame_time)
            
            if visual_input is not None and audio_input is not None:
                samples.append({
                    'video_name': video_name,
                    'frame_idx': frame_count,
                    'visual_input': visual_input,
                    'audio_input': audio_input,
                    'frame_time': frame_time
                })
        
        frame_count += 1
    
    cap.release()
    return samples

def process_video_wrapper(video_path: str, output_file: str, max_duration: float):
    """Wrapper for parallel processing that saves results immediately"""
    samples = process_video(video_path, max_duration)
    if not samples:
        return
        
    # Process and save features for this video
    with torch.no_grad():
        visual_inputs = {
            'pixel_values': torch.cat([x['visual_input']['pixel_values'] for x in samples])
        }
        audio_inputs = {
            'input_values': torch.cat([x['audio_input']['input_values'] for x in samples]),
            'attention_mask': torch.cat([x['audio_input']['attention_mask'] for x in samples])
        }
        
        visual_features, audio_features = feature_extractor(visual_inputs, audio_inputs)
        fused_features = fusion_model(visual_features, audio_features)
        
        for i in range(len(samples)):
            append_to_csv(output_file, {
                'video_file': samples[i]['video_name'],
                'frame_index': samples[i]['frame_idx'],
                'frame_time': samples[i]['frame_time'],
                'visual_features': visual_features[i].cpu().numpy().tolist(),
                'audio_features': audio_features[i].cpu().numpy().tolist(),
                'fused_features': fused_features[i].cpu().numpy().tolist()
            })

def process_folder(folder_path: str, output_file: str = "fused_av_features.csv", 
                  max_duration: float = 5.0, max_workers: int = 4) -> None:
    """Process all videos in folder with parallel execution"""
    video_paths = glob.glob(os.path.join(folder_path, '**', '*.mp4'), recursive=True)
    if not video_paths:
        print("No MP4 videos found in the specified folder.")
        return
    
    print(f"Found {len(video_paths)} videos to process")
    initialize_csv(output_file)
    
    # Process videos in parallel
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        list(tqdm(
            executor.map(
                partial(process_video_wrapper, output_file=output_file, max_duration=max_duration),
                video_paths
            ),
            total=len(video_paths),
            desc="Processing Videos"
        ))
    
    print(f"\nProcessing complete. Results saved to {output_file}")

if __name__ == "__main__":
    folder_path = "../Augmented/fake"  # Change to your folder path
    output_file = "ViT_fused_av_features_fake.csv"
    
    # Run with optimized parameters
    process_folder(
        folder_path,
        output_file=output_file,
        max_duration=5.0,
        max_workers=4  # Adjust based on your CPU cores
    )

Using device: NVIDIA GeForce RTX 4060 Laptop GPU
Found 13605 videos to process


Processing Videos:   0%|▎                                                        | 65/13605 [12:48<39:49:04, 10.59s/it]

WARNING NMS time limit 2.050s exceeded


Processing Videos:   2%|▉                                                       | 213/13605 [38:53<35:58:14,  9.67s/it]

WARNING NMS time limit 2.050s exceeded


...

WARNING NMS time limit 2.050s exceeded


Processing Videos: 100%|█████████████████████████████████████████████████████▊| 13551/13605 [46:35:34<11:19, 12.58s/it]

WARNING NMS time limit 2.050s exceeded


Processing Videos: 100%|██████████████████████████████████████████████████████| 13605/13605 [46:47:53<00:00, 12.38s/it]


Processing complete. Results saved to ViT_fused_av_features_fake.csv


In [5]:
import cv2
import mediapipe as mp
import os
import numpy as np
import pandas as pd
import glob
from transformers import ViTFeatureExtractor, ViTModel, Wav2Vec2FeatureExtractor, Wav2Vec2Model
import torch
import torch.nn as nn
import ffmpeg
import librosa
import noisereduce as nr
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
from torch.utils.data import Dataset, DataLoader
from typing import Tuple, Dict, List, Any
from tqdm import tqdm
import csv
import concurrent.futures
from functools import partial

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {torch.cuda.get_device_name(device)}")

# Initialize models globally for reuse
mp_face_mesh = mp.solutions.face_mesh

# Mouth landmark indices (MediaPipe's 468-point face mesh)
MOUTH_LANDMARKS = [
    61, 185, 40, 39, 37, 0, 267, 269, 270, 409,
    291, 146, 91, 181, 84, 17, 314, 405, 321, 375,
    78, 191, 80, 81, 82, 13, 312, 311, 310, 415,
    95, 88, 178, 87, 14, 317, 402, 318, 324, 308
]

# Initialize models with caching
class AVFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.visual_extractor = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k').to(device).eval()
        self.audio_extractor = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-xls-r-300m").to(device).eval()
        self.vit_feature_extractor = ViTFeatureExtractor.from_pretrained('google/vit-base-patch16-224-in21k')
        self.wav2vec_feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-xls-r-300m")
        
        # Freeze models
        for param in self.visual_extractor.parameters():
            param.requires_grad = False
        for param in self.audio_extractor.parameters():
            param.requires_grad = False
            
    def forward(self, visual_input, audio_input):
        with torch.no_grad():
            visual_features = self.visual_extractor(**visual_input).last_hidden_state.mean(dim=1)
            audio_features = self.audio_extractor(**audio_input).last_hidden_state.mean(dim=1)
        return visual_features, audio_features

class AVFusionModel(nn.Module):
    def __init__(self, visual_dim=768, audio_dim=1024, hidden_dim=512):
        super().__init__()
        self.fusion = nn.Sequential(
            nn.Linear(visual_dim + audio_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
    def forward(self, visual_features, audio_features):
        return self.fusion(torch.cat([visual_features, audio_features], dim=1))

# Initialize models
feature_extractor = AVFeatureExtractor()
fusion_model = AVFusionModel().to(device).eval()
yolo_face_model = YOLO(hf_hub_download(repo_id="arnabdhar/YOLOv8-Face-Detection", filename="model.pt"))

def initialize_csv(output_file: str):
    """Initialize CSV file with headers"""
    with open(output_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            'video_file', 'frame_index', 'frame_time',
            'visual_features', 'audio_features', 'fused_features'
        ])

def append_to_csv(output_file: str, data: Dict[str, Any]):
    """Append a single row to the CSV file"""
    with open(output_file, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            data['video_file'],
            data['frame_index'],
            data['frame_time'],
            str(data['visual_features']),  # Convert lists to strings for CSV
            str(data['audio_features']),
            str(data['fused_features'])
        ])

def get_mouth_roi(face_landmarks: Any, frame: np.ndarray, padding: int = 20) -> np.ndarray:
    """Optimized mouth region extraction"""
    h, w = frame.shape[:2]
    coords = np.array([(int(lm.x * w), int(lm.y * h)) for lm in [face_landmarks.landmark[i] for i in MOUTH_LANDMARKS]])
    x_min, y_min = np.min(coords, axis=0) - padding
    x_max, y_max = np.max(coords, axis=0) + padding
    return frame[max(0,y_min):min(h,y_max), max(0,x_min):min(w,x_max)]

def extract_visual_features(frame: np.ndarray) -> Dict[str, torch.Tensor]:
    """Optimized visual feature extraction with batching support"""
    try:
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Initialize FaceMesh for each frame to avoid timestamp issues
        with mp_face_mesh.FaceMesh(
            static_image_mode=True,  # Treat each frame independently
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        ) as face_mesh:
            results = face_mesh.process(image_rgb)
            
            if results.multi_face_landmarks:
                mouth_roi = get_mouth_roi(results.multi_face_landmarks[0], frame)
                if mouth_roi.size > 0:
                    mouth_roi = cv2.resize(mouth_roi, (224, 224))
                    return feature_extractor.vit_feature_extractor(
                        images=cv2.cvtColor(mouth_roi, cv2.COLOR_BGR2RGB), 
                        return_tensors="pt"
                    ).to(device)
                    
        # Fallback to YOLO if MediaPipe fails
        results = yolo_face_model.predict(frame, conf=0.5, verbose=False, imgsz=320)
        if results and results[0].boxes is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            if len(boxes) > 0:
                x1, y1, x2, y2 = boxes[0].astype(int)
                cropped_face = frame[y1:y2, x1:x2]
                if cropped_face.size > 0:
                    with mp_face_mesh.FaceMesh(
                        static_image_mode=True,
                        max_num_faces=1,
                        refine_landmarks=True,
                        min_detection_confidence=0.5,
                        min_tracking_confidence=0.5
                    ) as face_mesh:
                        results = face_mesh.process(cv2.cvtColor(cropped_face, cv2.COLOR_BGR2RGB))
                        if results.multi_face_landmarks:
                            mouth_roi = get_mouth_roi(results.multi_face_landmarks[0], cropped_face)
                            if mouth_roi.size > 0:
                                mouth_roi = cv2.resize(mouth_roi, (224, 224))
                                return feature_extractor.vit_feature_extractor(
                                    images=cv2.cvtColor(mouth_roi, cv2.COLOR_BGR2RGB),
                                    return_tensors="pt"
                                ).to(device)
        return None
    except Exception as e:
        print(f"\nVisual processing error: {str(e)}")
        return None

def extract_audio_features(audio: np.ndarray, sr: int, frame_time: float, 
                         window_size: float = 0.2) -> Dict[str, torch.Tensor]:
    """Optimized audio feature extraction"""
    try:
        start = int(max(0, (frame_time - window_size/2) * sr))
        end = int(min(len(audio), (frame_time + window_size/2) * sr))
        segment = audio[start:end] if (end-start) > 0 else np.zeros(int(sr * window_size))
        
        return feature_extractor.wav2vec_feature_extractor(
            segment,
            return_tensors="pt",
            sampling_rate=sr,
            padding="max_length",
            max_length=int(sr * window_size),
            truncation=True
        ).to(device)
    except Exception as e:
        print(f"\nAudio processing error: {str(e)}")
        return None

def extract_audio(video_path: str) -> Tuple[np.ndarray, int]:
    """Faster audio extraction using FFmpeg pipe"""
    try:
        out, _ = (
            ffmpeg.input(video_path)
            .output('pipe:', format='wav', ac=1, ar=16000)
            .run(capture_stdout=True, quiet=True)
        )
        return np.frombuffer(out, np.int16).astype(np.float32) / 32768.0, 16000
    except Exception as e:
        print(f"\nError extracting audio: {str(e)}")
        return None, None

def process_video(video_path: str, max_duration: float = 5.0) -> List[Dict[str, Any]]:
    """Process a single video with optimized pipeline"""
    video_name = os.path.basename(video_path)
    samples = []
    
    # Extract audio once per video
    audio, sr = extract_audio(video_path)
    if audio is None:
        return []
    
    # Apply noise reduction to entire audio
    try:
        audio = nr.reduce_noise(y=audio, sr=sr)
    except Exception as e:
        print(f"\nNoise reduction failed: {str(e)}")
    
    # Process video frames
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0:
        cap.release()
        return []
    
    frame_interval = max(1, int(fps / 24))  # Target ~24fps
    max_frame = int(fps * max_duration)
    frame_count = 0
    
    while cap.isOpened() and frame_count < max_frame:
        ret, frame = cap.read()
        if not ret:
            break
            
        if frame_count % frame_interval == 0:
            frame_time = frame_count / fps
            visual_input = extract_visual_features(frame)
            audio_input = extract_audio_features(audio, sr, frame_time)
            
            if visual_input is not None and audio_input is not None:
                samples.append({
                    'video_name': video_name,
                    'frame_idx': frame_count,
                    'visual_input': visual_input,
                    'audio_input': audio_input,
                    'frame_time': frame_time
                })
        
        frame_count += 1
    
    cap.release()
    return samples

def process_video_wrapper(video_path: str, output_file: str, max_duration: float):
    """Wrapper for parallel processing that saves results immediately"""
    samples = process_video(video_path, max_duration)
    if not samples:
        return
        
    # Process and save features for this video
    with torch.no_grad():
        visual_inputs = {
            'pixel_values': torch.cat([x['visual_input']['pixel_values'] for x in samples])
        }
        audio_inputs = {
            'input_values': torch.cat([x['audio_input']['input_values'] for x in samples]),
            'attention_mask': torch.cat([x['audio_input']['attention_mask'] for x in samples])
        }
        
        visual_features, audio_features = feature_extractor(visual_inputs, audio_inputs)
        fused_features = fusion_model(visual_features, audio_features)
        
        for i in range(len(samples)):
            append_to_csv(output_file, {
                'video_file': samples[i]['video_name'],
                'frame_index': samples[i]['frame_idx'],
                'frame_time': samples[i]['frame_time'],
                'visual_features': visual_features[i].cpu().numpy().tolist(),
                'audio_features': audio_features[i].cpu().numpy().tolist(),
                'fused_features': fused_features[i].cpu().numpy().tolist()
            })

def process_folder(folder_path: str, output_file: str = "fused_av_features.csv", 
                  max_duration: float = 5.0, max_workers: int = 4) -> None:
    """Process all videos in folder with parallel execution"""
    video_paths = glob.glob(os.path.join(folder_path, '**', '*.mp4'), recursive=True)
    if not video_paths:
        print("No MP4 videos found in the specified folder.")
        return
    
    print(f"Found {len(video_paths)} videos to process")
    initialize_csv(output_file)
    
    # Process videos in parallel
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        list(tqdm(
            executor.map(
                partial(process_video_wrapper, output_file=output_file, max_duration=max_duration),
                video_paths
            ),
            total=len(video_paths),
            desc="Processing Videos"
        ))
    
    print(f"\nProcessing complete. Results saved to {output_file}")

if __name__ == "__main__":
    folder_path = "../Augmented/real"  # Change to your folder path
    output_file = "ViT_fused_av_features_real.csv"
    
    # Run with optimized parameters
    process_folder(
        folder_path,
        output_file=output_file,
        max_duration=5.0,
        max_workers=4  # Adjust based on your CPU cores
    )

Using device: NVIDIA GeForce RTX 4060 Laptop GPU
Found 5020 videos to process


Processing Videos:   0%|                                                                      | 0/5020 [00:00<?, ?it/s]

Ultralytics 8.3.136  Python-3.12.3 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Ultralytics 8.3.136  Python-3.12.3 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


Processing Videos:   0%|▎                                                          | 24/5020 [03:30<9:57:47,  7.18s/it]

WARNING NMS time limit 2.050s exceeded


...

WARNING NMS time limit 2.050s exceeded


Processing Videos:   2%|█▎                                                        | 116/5020 [20:57<8:08:56,  5.98s/it]

WARNING NMS time limit 2.050s exceeded


Processing Videos:   2%|█▍                                                       | 123/5020 [23:14<15:25:32, 11.34s/it]


OSError: [Errno 28] No space left on device